# Fraud Dectection Model

## 1. Install Libraries

In [0]:
# Installing Libraries
!pip install numpy pandas matplotlib seaborn scikit-learn

In [0]:
pip install pandas sqlalchemy


In [0]:
pip install sweetviz

## 2. Importing libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import plotly.express as px
from   plotly.offline	import	iplot
import plotly.graph_objects as go
from   plotly.subplots	import	make_subplots
import plotly.figure_factory as ff
import sweetviz as sv




In [0]:
import warnings
warnings.filterwarnings("ignore") 
sns.set(style="whitegrid")


## 3. Data Ingestion

In [0]:
# 1. Read the table into a Spark DataFrame
spark_bank_fraud = spark.read.table("workspace.mlops.gold_nibs")

# 2. Convert it to a traditional Pandas DataFrame
bank_fraud = spark_bank_fraud.toPandas()

# View your data
display(bank_fraud.head(2))

## 4. Data Overview

In [0]:
# Reads the top 5 rows
display(bank_fraud.head(5))

## 5. EXPLORATORY DATA ANALYSIS

### 5.1 Data Inspection & Understanding

In [0]:
#Understanding the shape like the size or dimension

bank_fraud.shape


In [0]:
# Which Coloumn have null, Column data types, Nummber Rows, Numbr Of columns and memory usage

bank_fraud.info()     

# Timestamp needs to be converted to timestamp data type, fraud technique has null, must be handle with unknown

In [0]:
# Count of Rows which have a null valaue of is blank Per Column.
bank_fraud.isnull().sum()   # Fraud_Technique  has 997000 nulls



##### Selecting uniques locations and channel, merchant category, bank, age group

In [0]:
cols = ['bank', 'merchant_category', 'channel', 'location', 'age_group']

unique_vals = {col: bank_fraud[col].unique().tolist() for col in cols}

for col, vals in unique_vals.items():
    print(f"\n{col.upper()} ({len(vals)} unique): {vals}")

In [0]:
# Checking Is_Fraud Distribution

bank_fraud['is_fraud'].value_counts()
 

In [0]:
# Shows nulls per Column
bank_fraud.isnull().sum()

In [0]:
# Shows overall nulls in a dataset
bank_fraud.isnull().sum().sum()

In [0]:
# Check for duplicates
bank_fraud.duplicated().sum()

In [0]:
display(bank_fraud.head(5))

In [0]:
bank_fraud.describe().T

In [0]:
# Data Quality Checks
print("\n=== Amount ===")
print(f"Min : NGN {bank_fraud['amount'].min():,.2f}")
print(f"Max : NGN {bank_fraud['amount'].max():,.2f}")
neg = (bank_fraud['amount'] < 0).sum()
print(f"Negative values : {neg}  ({neg/len(bank_fraud)*100:.2f}%)")

In [0]:
# To prevent changing raw data
bank_fraud_clean =bank_fraud.copy()

In [0]:
# Check for duplicates
bank_fraud_clean.duplicated().sum()

### 5.2. Data Cleaning
  #### > Timestamp srt to date
  #### > Remove nulls, replace with unknown

In [0]:
# Converting timestamp string to datetime
bank_fraud_clean['timestamp'] = pd.to_datetime(bank_fraud_clean['timestamp'],format='%Y-%m-%d %H:%M:%S')

In [0]:
bank_fraud['timestamp'].dtype
print(type(bank_fraud_clean['timestamp']))
print(bank_fraud_clean['timestamp'].dtype)

## 6. Feature Engineering

In [0]:
#Creating Date Features

bank_fraud_clean['year'] = bank_fraud_clean['timestamp'].dt.year
bank_fraud_clean['month'] = bank_fraud_clean['timestamp'].dt.month_name()
bank_fraud_clean['day_of_week'] = bank_fraud_clean['timestamp'].dt.day_name()
bank_fraud_clean['hour'] = bank_fraud_clean['timestamp'].dt.hour
bank_fraud_clean['time'] = pd.to_datetime(bank_fraud_clean['timestamp']).dt.time
bank_fraud_clean['day'] = bank_fraud_clean['timestamp'].dt.day

In [0]:
bank_fraud_clean.head(3)

In [0]:
bank_fraud_clean.columns

In [0]:
# Creating Day Classification : Weekday Or Weekend 
bank_fraud_clean['DayClassification'] = 'Weekday'  # Set a default first
bank_fraud_clean.loc[bank_fraud_clean['day'] == 'Sunday', 'DayClassification'] = 'Weekend'
bank_fraud_clean.loc[bank_fraud_clean['day'] == 'Saturday', 'DayClassification'] = 'Weekend'

bank_fraud_clean.head(3)

In [0]:
# Create Season Category
bank_fraud_clean['season'] = 'Unknown'

bank_fraud_clean.loc[bank_fraud_clean['month'].isin(['November', 'December', 'January', 'February', 'March']), 'season'] = 'Dry Season'
bank_fraud_clean.loc[bank_fraud_clean['month'].isin(['April', 'May', 'June', 'July', 'August', 'September', 'October']), 'season'] = 'Rainy Season'

print(bank_fraud_clean['season'].value_counts())

In [0]:
bank_fraud_clean = bank_fraud_clean.drop(
    columns=[
        'fraud_technique', 'tx_count_24h', 'amount_sum_24h',
        'amount_mean_7d', 'amount_std_7d', 'tx_count_total',
        'amount_mean_total', 'amount_std_total',
        'channel_diversity', 'location_diversity',
        'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
        'month_sin', 'month_cos', 'amount_rounded'
    ],
    errors='ignore'
)

In [0]:
cols_to_drop = [
    'fraud_technique', 'tx_count_24h', 'amount_sum_24h',
    'amount_mean_7d', 'amount_std_7d', 'tx_count_total',
    'amount_mean_total', 'amount_std_total',
    'channel_diversity', 'location_diversity',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'month_sin', 'month_cos', 'amount_rounded'
]

print(type(bank_fraud_clean))

In [0]:

bank_fraud_clean = bank_fraud_clean.drop(cols_to_drop, errors='ignore')

In [0]:
bank_fraud_clean.shape

In [0]:
bank_fraud_clean = bank_fraud_clean.drop(['transaction_hour_category','amount_vs_mean_ratio','transaction_id', 'customer_id'], axis=1, errors='ignore')

In [0]:
# Preview if The columns were dropped
bank_fraud_clean.head(3)

In [0]:
# Determine Time Range For Time Buckets
bank_fraud_clean['time'].min()
bank_fraud_clean['time'].max()

In [0]:
# Create Time Buckets

bank_fraud_clean['time_bucket'] = 'Unknown'

bank_fraud_clean.loc[bank_fraud_clean['hour'].between(0, 4),   'time_bucket'] = 'Midnight (00:00–04:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(5, 7),   'time_bucket'] = 'Early Morning (05:00–07:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(8, 11),  'time_bucket'] = 'Morning (08:00–11:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(12, 14), 'time_bucket'] = 'Midday (12:00–14:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(15, 17), 'time_bucket'] = 'Afternoon (15:00–17:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(18, 20), 'time_bucket'] = 'Evening (18:00–20:59)'
bank_fraud_clean.loc[bank_fraud_clean['hour'].between(21, 23), 'time_bucket'] = 'Night (21:00–23:59)'

print(bank_fraud_clean['time_bucket'].value_counts())

In [0]:
# MONTH CATEGORY
bank_fraud_clean['month_category'] = 'Unknown'

bank_fraud_clean.loc[bank_fraud_clean['day'].between(1, 10),  'month_category'] = 'Beginning (1st–10th)'
bank_fraud_clean.loc[bank_fraud_clean['day'].between(11, 20), 'month_category'] = 'Mid Month (11th–20th)'
bank_fraud_clean.loc[bank_fraud_clean['day'].between(21, 31), 'month_category'] = 'Month End (21st–31st)'

print(bank_fraud_clean['month_category'].value_counts())

In [0]:
#Preview 
bank_fraud_clean.head(3)

# Visualizing

#### Phase 1

In [0]:
# Understanding Imbalance
# A very low fraud rate means accuracy is not an appropriate evaluation metric
#fraud_rate = bank_fraud_clean['is_fraud'].mean()*100
#print(fraud_rate)

fraud_rate = bank_fraud_clean['is_fraud'].mean() * 100
print(f"Fraud Rate: {fraud_rate:.2f}%")

In [0]:


# Fraud vs Non Fraud Distribution using Plotly
fraud_counts = bank_fraud_clean['is_fraud'].value_counts().reset_index()
fraud_counts.columns = ['is_fraud', 'count']

fig = px.bar(
    fraud_counts,
    x='is_fraud',
    y='count',
    color='is_fraud',
    text='count',
    labels={'is_fraud': 'Is Fraud_Not Fraud', 'count': 'Count'},
    title='Fraud(1) vs Non-Fraud Count(0)'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis=dict(tickmode='array', tickvals=[0, 1]),
    yaxis_title='Count',
    xaxis_title='Is Fraud_Not Fraud',
    title_x=0.5
)

fig.show()

In [0]:

# Fraud vs Non-fraud count using Plotly
fraud_counts = bank_fraud_clean['is_fraud'].value_counts().reset_index()
fraud_counts.columns = ['is_fraud', 'count']

fig = px.pie(
    fraud_counts,
    values='count',
    names='is_fraud',
    color='is_fraud',
    title="Fraud vs Non-Fraud Transactions",
    hole=0.4
)

fig.update_traces(textinfo='percent+label')
fig.update_layout(title_x=0.5)
fig.show()

In [0]:


# Transaction Amount Distribution by Fraud Status using Plotly
fig = px.histogram(
    bank_fraud_clean,
    x='amount',
    color='is_fraud',
    nbins=50,
    barmode='overlay',
    histnorm='density',
    title='Transaction Amount Distribution by Fraud Status'
)

fig.update_layout(
    xaxis_title='Transaction Amount',
    yaxis_title='Density',
    title_x=0.5
)

fig.show()

In [0]:
# Transaction Amount Distribution, Fixing the skeweness (Used Log Transoformation to represent a bell shape)
fig = px.histogram(
    bank_fraud_clean,
    x='amount_log',
    nbins=50,
    histnorm='density',
    title='Distribution of Log-Transformed Transaction Amounts'
)

fig.update_layout(
    xaxis_title='Log(Transaction Amount)',
    yaxis_title='Density',
    title_x=0.5
)

fig.show()

In [0]:
# Transaction Amount Distribution, Fixing the skeweness (Used Log Transoformation to represent a bell shape) added a legend Is fraud(0 or 1)
fig = px.histogram(
    bank_fraud_clean,
    x='amount_log',
    color='is_fraud',
    nbins=20,
    histnorm='density',
    barmode='overlay',
    title='Log-Transformed Amount Distribution by Fraud Status'
)

fig.update_layout(
    xaxis_title='Log(Transaction Amount)',
    yaxis_title='Density',
    title_x=0.5,
    legend_title_text='Is Fraud'
)

fig.show()

In [0]:
# Count transactions per day
daily_transactions = (
    bank_fraud_clean
    .groupby(bank_fraud_clean['timestamp'].dt.date)
    .size()
    .reset_index(name='transaction_count')
)

daily_transactions.columns = ['date', 'transaction_count']

fig = px.line(
    daily_transactions,
    x='date',
    y='transaction_count',
    markers=True,
    title='Daily Transaction Volume Trend'
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Number of Transactions',
    title_x=0.5
)

fig.show()

### Phase 2

In [0]:
# Fraud Rate by Channel
bank_fraud_clean.groupby('channel')['is_fraud'].mean()*100

#Identifies risky transaction channels.
#Validates whether channel is predictive.

In [0]:
fraud_channel = (
    bank_fraud_clean.groupby('channel')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
    .sort_values('is_fraud', ascending=False)
)

fig = px.bar(
    fraud_channel,
    x='is_fraud',
    y='channel',
    orientation='h',
    text='is_fraud',
    labels={'is_fraud': 'Fraud Rate (%)', 'channel': 'Channel'},
    title='Fraud Rate by Channel'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Fraud Rate (%)',
    yaxis_title='Channel',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Merchant Category

bank_fraud_clean.groupby('merchant_category')['is_fraud'].mean()*100

# Why?
# Some merchant categories naturally attract more fraud.

# Model Impact
# Strong categorical feature.

In [0]:
# Fraud Rate BY Merchant Category
fraud_merchant = (
    bank_fraud_clean.groupby('merchant_category')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
    .sort_values('is_fraud', ascending=False)
)

fig = px.bar(
    fraud_merchant,
    x='is_fraud',
    y='merchant_category',
    orientation='h',
    text='is_fraud',
    labels={'is_fraud': 'Fraud Rate (%)', 'merchant_category': 'Merchant Category'},
    title='Fraud Rate by Merchant Category'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Fraud Rate (%)',
    yaxis_title='Merchant Category',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Bank
bank_fraud_clean.groupby('bank')['is_fraud'].mean()*100

# Detects institutions experiencing higher fraud exposure.

In [0]:
# Fraud Rate BY Bank

fraud_bank = (
    bank_fraud_clean.groupby('bank')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
    .sort_values('is_fraud', ascending=False)
)

fig = px.bar(
    fraud_bank,
    x='is_fraud',
    y='bank',
    orientation='h',
    text='is_fraud',
    labels={'is_fraud': 'Fraud Rate (%)', 'bank': 'Bank'},
    title='Fraud Rate by Bank'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Fraud Rate (%)',
    yaxis_title='Bank',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Location

bank_fraud_clean.groupby('location')['is_fraud'].mean()*100
# Certain regions may have elevated fraud activity

In [0]:
# Fraud Rate By Location
fraud_location = (
    bank_fraud_clean.groupby('location')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
    .sort_values('is_fraud', ascending=False)
)

fig = px.bar(
    fraud_location,
    x='is_fraud',
    y='location',
    orientation='h',
    text='is_fraud',
    labels={'is_fraud': 'Fraud Rate (%)', 'location': 'Location'},
    title='Fraud Rate by Location'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Fraud Rate (%)',
    yaxis_title='Location',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Age Group

bank_fraud_clean.groupby('age_group')['is_fraud'].mean()*100

# Determines whether age influences fraud likelihood.

In [0]:
fraud_age = (
    bank_fraud_clean.groupby('age_group')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.bar(
    fraud_age,
    x='is_fraud',
    y='age_group',
    orientation='h',
    text='is_fraud',
    labels={'is_fraud': 'Fraud Rate (%)', 'age_group': 'Age Group'},
    title='Fraud Rate by Age Group'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Fraud Rate (%)',
    yaxis_title='Age Group',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Hour

bank_fraud_clean.groupby('hour')['is_fraud'].mean()*100

# Fraud often occurs at unusual hours.


In [0]:
# Hourly Fraud Rates
fraud_hour = (
    bank_fraud_clean.groupby('hour')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    fraud_hour,
    x='hour',
    y='is_fraud',
    markers=True,
    labels={'hour': 'Hour of Day', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Hour'
)

fig.update_traces(text=[f'{y:.2f}%' for y in fraud_hour['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Hour of Day',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    yaxis_gridcolor='rgba(0,0,0,0.1)'
)

fig.show()

In [0]:
# Fraud Rate by Day of Week

bank_fraud_clean.groupby('day_of_week')['is_fraud'].mean()*100

# Fraud patterns may vary across the week.



In [0]:
# Daily Fraud Rate

fraud_day = (
    bank_fraud_clean.groupby('day_of_week')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    fraud_day,
    x='day_of_week',
    y='is_fraud',
    markers=True,
    labels={'day_of_week': 'Day of Week', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Day of Week'
)

fig.update_traces(text=[f'{y:.2f}%' for y in fraud_day['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Day of Week',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    yaxis_gridcolor='rgba(0,0,0,0.1)'
)

fig.show()

In [0]:
# Fraud Rate by Month

bank_fraud_clean.groupby('month')['is_fraud'].mean()*100

# Detects seasonality.


In [0]:
# Monthly Fraud Rate
fraud_month = (
    bank_fraud_clean.groupby('month')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

# Ensure months are in calendar order
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

fraud_month['month'] = pd.Categorical(
    fraud_month['month'],
    categories=month_order,
    ordered=True
)

fraud_month = fraud_month.sort_values('month')

fig = px.line(
    fraud_month,
    x='month',
    y='is_fraud',
    markers=True,
    labels={'month': 'Month', 'is_fraud': 'Fraud Rate (%)'},
    title='Monthly Fraud Rate Trend'
)

fig.update_traces(text=[f'{y:.2f}%' for y in fraud_month['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5
)

fig.show()

In [0]:
# Count transactions by season and month
transactions = (
    bank_fraud_clean
    .groupby(['season', 'month'])
    .size()
    .reset_index(name='transaction_count')
)

transactions = transactions.sort_values('month')

fig = px.line(
    transactions,
    x='month',
    y='transaction_count',
    color='season',
    markers=True,
    labels={'month': 'Month', 'transaction_count': 'Number of Transactions', 'season': 'Season'},
    title='Transaction Volume Over Time by Season'
)

fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Number of Transactions',
    title_x=0.5,
    legend_title_text='Season'
)

fig.show()

In [0]:
# Seasonal Fraud Distribution
seasonal_fraud = (
    bank_fraud_clean.groupby('season')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.bar(
    seasonal_fraud,
    x='season',
    y='is_fraud',
    text='is_fraud',
    labels={'season': 'Season', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Season'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Season',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5
)

fig.show()

### Phase 3

In [0]:
# Amount Boxplot by Fraud Status

fig = px.box(
    bank_fraud_clean,
    x='is_fraud',
    y='amount',
    points='outliers',
    labels={'is_fraud': 'Fraud Status', 'amount': 'Transaction Amount'},
    title='Amount Boxplot by Fraud Status'
)
fig.update_layout(title_x=0.5)
fig.show()
# Quickly identifies outlier behavior.

In [0]:
# Velocity Score Analysis

fig = px.box(
    bank_fraud_clean,
    x='is_fraud',
    y='velocity_score',
    points='outliers',
    labels={'is_fraud': 'Fraud Status', 'velocity_score': 'Velocity Score'},
    title='Velocity Score by Fraud Status'
)
fig.update_layout(title_x=0.5)
fig.show()

#Velocity is often one of the strongest fraud predictors.

In [0]:
# Merchant Risk Score Analysis

fig = px.box(
    bank_fraud_clean,
    x='is_fraud',
    y='merchant_risk_score',
    points='outliers',
    labels={'is_fraud': 'Fraud Status', 'merchant_risk_score': 'Merchant Risk Score'},
    title='Merchant Risk Score by Fraud Status'
)
fig.update_layout(title_x=0.5)
fig.show()

# Confirms whether fraudulent transactions occur with higher-risk merchants.

In [0]:
# Composite Risk Score Analysis

fig = px.box(
    bank_fraud_clean,
    x='is_fraud',
    y='composite_risk',
    points='outliers',
    labels={'is_fraud': 'Fraud Status', 'composite_risk': 'Composite Risk Score'},
    title='Composite Risk Score by Fraud Status'
)
fig.update_layout(title_x=0.5)
fig.show()

# This engineered feature should strongly separate fraud from non-fraud.

In [0]:
#Correlation Heatmap

numeric_cols = bank_fraud_clean.select_dtypes(include=np.number)

fig = px.imshow(
    numeric_cols.corr(),
    text_auto=True,
    color_continuous_scale='RdBu',
    labels={'color': 'Correlation'},
    title='Correlation Heatmap'
)

fig.update_layout(title_x=0.5, width=900, height=700)
fig.show()

# Detects:multicollinearity, redundant features

In [0]:
# Correlation with Fraud

corr_target = (
    bank_fraud_clean.select_dtypes(include='number')
    .corr()['is_fraud']
    .sort_values(ascending=False)
    .drop('is_fraud')
    .reset_index()
)

corr_target.columns = ['feature', 'correlation']

fig = px.bar(
    corr_target,
    x='correlation',
    y='feature',
    orientation='h',
    text='correlation',
    labels={'correlation': 'Correlation Coefficient', 'feature': 'Feature'},
    title='Feature Correlation with Fraud'
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(title_x=0.5, xaxis_title='Correlation Coefficient', yaxis_title='Feature', height=600)
fig.show()

In [0]:
# Fraud Rate by Transaction Velocity

bank_fraud_clean['velocity_decile'] = pd.qcut(
    bank_fraud_clean['velocity_score'],
    10,
    labels=range(1, 11),
    duplicates='drop'
)

velocity_risk = (
    bank_fraud_clean.groupby('velocity_decile')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    velocity_risk,
    x='velocity_decile',
    y='is_fraud',
    markers=True,
    labels={'velocity_decile': 'Velocity Decile', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Transaction Velocity Decile'
)

fig.update_traces(text=[f'{y:.2f}%' for y in velocity_risk['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Velocity Decile',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5
)

fig.show()

# Determines whether risk increases with velocity.

In [0]:
# Merchant Risk Vs Velocity Score

fig = px.scatter(
    bank_fraud_clean.sample(min(10000, len(bank_fraud_clean))),
    x='merchant_risk_score',
    y='velocity_score',
    color='is_fraud',
    opacity=0.6,
    labels={'merchant_risk_score': 'Merchant Risk Score', 'velocity_score': 'Velocity Score', 'is_fraud': 'Fraud Status'},
    title='Merchant Risk Score vs Velocity Score'
)

fig.update_layout(title_x=0.5, width=900, height=600)
fig.show()

In [0]:
# Pairplot of Top Features

#Useful before feature selection.

top_features = [
    'amount',
    'velocity_score',
    'merchant_risk_score',
    'composite_risk',
    'is_fraud'
]

fig = px.scatter_matrix(
    bank_fraud_clean[top_features],
    dimensions=top_features[:-1],
    color='is_fraud',
    labels={col: col.replace('_', ' ').title() for col in top_features},
    title='Pairplot of Top Features'
)
fig.update_layout(title_x=0.5, width=900, height=900)
fig.show()

### Phase 4

In [0]:
# Fraud Rate by Composite Risk Deciles

bank_fraud_clean['risk_decile'] = pd.qcut(
    bank_fraud_clean['composite_risk'],
    q=10,
    labels=range(1, 11),
    duplicates='drop'
)

risk_profile = (
    bank_fraud_clean.groupby('risk_decile')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    risk_profile,
    x='risk_decile',
    y='is_fraud',
    markers=True,
    labels={'risk_decile': 'Risk Decile', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Composite Risk Decile'
)

fig.update_traces(text=[f'{y:.2f}%' for y in risk_profile['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Risk Decile',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5
)

fig.show()

In [0]:
# Fraud Rate by Transaction Amount Deciles
# Shows whether fraud increases as transaction amounts increase.

bank_fraud_clean['amount_decile'] = pd.qcut(
    bank_fraud_clean['amount'],
    10,
    labels=False,
    duplicates='drop'
)

amount_risk = (
    bank_fraud_clean.groupby('amount_decile')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    amount_risk,
    x='amount_decile',
    y='is_fraud',
    markers=True,
    labels={'amount_decile': 'Amount Decile', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate by Transaction Amount Decile'
)

fig.update_traces(text=[f'{y:.2f}%' for y in amount_risk['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Amount Decile',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5
)

fig.show()

# Cross-cut Analysis

In [0]:
bank_fraud.head(2)

In [0]:
# Bank × Merchant Category (Fraud Heat Pattern)
# Identifies which banks are exposed to fraud in specific merchant sectors

bank_merchant = (
    bank_fraud_clean.groupby(['bank', 'merchant_category'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

pivot_bank_merchant = bank_merchant.pivot(
    index='merchant_category',
    columns='bank',
    values='is_fraud'
).fillna(0)

fig = px.imshow(
    pivot_bank_merchant,
    labels={'x': 'Bank', 'y': 'Merchant Category', 'color': 'Fraud Rate (%)'},
    color_continuous_scale='Reds',
    text_auto=True,
    title='Fraud Rate: Bank vs Merchant Category'
)

fig.update_layout(title_x=0.5, width=1200, height=900)
fig.show()

In [0]:
# Time Risk Pattern (Hour × Weekend)
# Fraud is highly time-dependent (bots, automation, late-night activity)

time_risk = (
    bank_fraud_clean.groupby(['hour', 'is_weekend'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    time_risk,
    x='hour',
    y='is_fraud',
    color='is_weekend',
    markers=True,
    labels={'hour': 'Hour', 'is_fraud': 'Fraud Rate (%)', 'is_weekend': 'Weekend'},
    title='Fraud Rate by Hour and Weekend'
)

fig.update_traces(text=[f'{y:.2f}%' for y in time_risk['is_fraud']], textposition='top center')
fig.update_layout(
    xaxis_title='Hour',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1200,
    height=600
)
fig.show()

In [0]:
# Merchant Risk Score Validation
# WHY: Checks if your internal risk scoring system actually predicts fraud

merchant_risk = (
    bank_fraud_clean.groupby('merchant_risk_score')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    merchant_risk,
    x='merchant_risk_score',
    y='is_fraud',
    markers=True,
    labels={'merchant_risk_score': 'Merchant Risk Score', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate vs Merchant Risk Score'
)

fig.update_traces(text=[f"{y:.1f}%" for y in merchant_risk['is_fraud']], textposition='top center')
fig.update_layout(
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1000,
    height=500
)
fig.show()

In [0]:
# Composite Risk vs Fraud (MODEL VALIDATION)
# WHY: Validates if combined risk scoring aligns with real fraud outcomes

composite = (
    bank_fraud_clean.groupby('composite_risk')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    composite,
    x='composite_risk',
    y='is_fraud',
    markers=True,
    labels={'composite_risk': 'Composite Risk Score', 'is_fraud': 'Fraud Rate (%)'},
    title='Fraud Rate vs Composite Risk Score'
)

fig.update_traces(text=[f"{y:.1f}%" for y in composite['is_fraud']], textposition='top center')
fig.update_layout(title_x=0.5, width=1000, height=500)
fig.show()

In [0]:
# Velocity × Amount (Fraud Behavior Cluster)
# WHY: High velocity + high amount often indicates synthetic fraud behavior

velocity_amount = (
    bank_fraud_clean.groupby('velocity_score')['amount_log']
    .mean()
    .reset_index()
)

fig = px.scatter(
    velocity_amount,
    x='velocity_score',
    y='amount_log',
    labels={'velocity_score': 'Velocity Score', 'amount_log': 'Log Amount'},
    title='Velocity Score vs Transaction Amount'
)

fig.update_layout(title_x=0.5, width=900, height=500)
fig.show()

In [0]:
# Age Group × Channel
# WHY: Identifies demographic vulnerability patterns per channel

age_channel = (
    bank_fraud_clean.groupby(['age_group', 'channel'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.bar(
    age_channel,
    x='age_group',
    y='is_fraud',
    color='channel',
    text='is_fraud',
    labels={'age_group': 'Age Group', 'is_fraud': 'Fraud Rate (%)', 'channel': 'Channel'},
    title='Fraud Rate: Age Group vs Channel'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title='Fraud Rate (%)', title_x=0.5, width=900, height=500)
fig.show()

In [0]:
# Location × Merchant Category
# WHY: Detect geographic fraud clusters tied to specific merchant types

loc_merchant = (
    bank_fraud_clean.groupby(['location', 'merchant_category'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

pivot = loc_merchant.pivot(
    index='location',
    columns='merchant_category',
    values='is_fraud'
).fillna(0)

fig = px.imshow(
    pivot,
    labels={'x': 'Merchant Category', 'y': 'Location', 'color': 'Fraud Rate (%)'},
    color_continuous_scale='Blues',
    text_auto=True,
    title='Fraud Rate: Location vs Merchant Category'
)

fig.update_layout(title_x=0.5, width=1200, height=600)
fig.show()

In [0]:
# Time Bucket × Channel
# WHY: Detects when specific channels become risky

time_bucket = (
    bank_fraud_clean.groupby(['time_bucket', 'channel'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.bar(
    time_bucket,
    x='time_bucket',
    y='is_fraud',
    color='channel',
    text='is_fraud',
    labels={'time_bucket': 'Time Bucket', 'is_fraud': 'Fraud Rate (%)', 'channel': 'Channel'},
    title='Fraud Rate: Time Bucket vs Channel'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(title_x=0.5, width=900, height=500)
fig.show()

In [0]:
# Season × Amount Decile
# WHY: Checks seasonal fraud intensity across transaction sizes

season_amount = (
    bank_fraud_clean.groupby(['season', 'amount_decile'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

fig = px.line(
    season_amount,
    x='amount_decile',
    y='is_fraud',
    color='season',
    markers=True,
    labels={'amount_decile': 'Amount Decile', 'is_fraud': 'Fraud Rate (%)', 'season': 'Season'},
    title='Fraud Rate: Season vs Amount Decile'
)

fig.update_traces(text=[f"{y:.1f}%" for y in season_amount['is_fraud']], textposition='top center')
fig.update_layout(title_x=0.5, width=900, height=500)
fig.show()

In [0]:
# Customer Risk Distribution
# WHY: Shows how risk is distributed across customers (helps identify high-risk clusters)

customer_risk = (
    bank_fraud.groupby('customer_id')['composite_risk']
    .mean()
    .reset_index()
)

fig = px.histogram(
    customer_risk,
    x='composite_risk',
    nbins=30,
    marginal='box',
    labels={'composite_risk': 'Composite Risk Score'},
    title='Customer Risk Distribution (Composite Risk)'
)

fig.update_layout(
    xaxis_title='Composite Risk Score',
    yaxis_title='Number of Customers',
    title_x=0.5,
    bargap=0.05
)
fig.show()

#### Day Classification Fraud Rate

In [0]:
# Calculate DayClassification fraud rate
DayClassification_fraud_rate = (
    bank_fraud_clean
    .groupby('DayClassification')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

DayClassification_fraud_rate = DayClassification_fraud_rate.sort_values('DayClassification')

fig = px.bar(
    DayClassification_fraud_rate,
    x='DayClassification',
    y='is_fraud',
    text='is_fraud',
    labels={'DayClassification': 'Day Classification', 'is_fraud': 'Fraud Rate (%)'},
    title='Daily Fraud Rate Trend'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Day Classification',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1200,
    height=600
)
fig.show()

In [0]:
# Calculate DayClassification fraud rate
DayClassification_fraud_rate = (
    bank_fraud_clean
    .groupby('DayClassification')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

DayClassification_fraud_rate = DayClassification_fraud_rate.sort_values('DayClassification')

fig = px.bar(
    DayClassification_fraud_rate,
    x='DayClassification',
    y='is_fraud',
    text='is_fraud',
    labels={'DayClassification': 'Day Classification', 'is_fraud': 'Fraud Rate (%)'},
    title='Daily Fraud Rate Trend'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    xaxis_title='Day Classification',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1200,
    height=600
)
fig.show()

#### Day Classification Fraud Rate Split by Segment Customised Colors

In [0]:
segments = ['bank', 'location', 'channel', 'merchant_category']

for segment in segments:
    segment_fraud_rate = (
        bank_fraud_clean
        .groupby(['DayClassification', segment])['is_fraud']
        .mean()
        .mul(100)
        .reset_index()
    )
    segment_fraud_rate = segment_fraud_rate.sort_values('DayClassification')

    fig = px.bar(
        segment_fraud_rate,
        x='DayClassification',
        y='is_fraud',
        color=segment,
        text='is_fraud',
        labels={
            'DayClassification': 'Day Classification',
            'is_fraud': 'Fraud Rate (%)',
            segment: segment.replace("_", " ").title()
        },
        title=f'Daily Fraud Rate by {segment.replace("_", " ").title()}'
    )

    fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
    fig.update_layout(
        xaxis_title='Day Classification',
        yaxis_title='Fraud Rate (%)',
        title_x=0.5,
        width=1200,
        height=600
    )
    fig.show()

#### Time Buckets: Time of the day with high Fraud Rate

In [0]:
segments = ['bank', 'location', 'channel', 'merchant_category']

fig = make_subplots(rows=2, cols=2, subplot_titles=[f'Daily Fraud Rate by {seg.replace("_", " ").title()}' for seg in segments])

for idx, segment in enumerate(segments):
    segment_fraud_rate = (
        bank_fraud_clean
        .groupby(['time_bucket', segment])['is_fraud']
        .mean()
        .mul(100)
        .reset_index()
    )
    segment_fraud_rate = segment_fraud_rate.sort_values('time_bucket')

    for cat in segment_fraud_rate[segment].unique():
        df_cat = segment_fraud_rate[segment_fraud_rate[segment] == cat]
        fig.add_trace(
            go.Bar(
                x=df_cat['time_bucket'],
                y=df_cat['is_fraud'],
                name=str(cat),
                text=[f'{v:.2f}%' for v in df_cat['is_fraud']],
                textposition='outside'
            ),
            row=idx // 2 + 1,
            col=idx % 2 + 1
        )

fig.update_layout(
    title='Daily Fraud Rate by Segment',
    title_x=0.5,
    height=900,
    width=1200,
    showlegend=True
)
fig.show()

In [0]:
segments = ['bank', 'location', 'channel', 'merchant_category']

for idx, segment in enumerate(segments):
    segment_fraud_rate = (
        bank_fraud_clean
        .groupby(['time_bucket', segment])['is_fraud']
        .mean()
        .mul(100)
        .reset_index()
    )
    segment_fraud_rate = segment_fraud_rate.sort_values('time_bucket')

    fig = px.bar(
        segment_fraud_rate,
        x='time_bucket',
        y='is_fraud',
        color=segment,
        text='is_fraud',
        labels={
            'time_bucket': 'Time Bucket',
            'is_fraud': 'Fraud Rate (%)',
            segment: segment.replace("_", " ").title()
        },
        title=f'Fraud Rate by Time Bucket and {segment.replace("_", " ").title()}'
    )

    fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
    fig.update_layout(
        xaxis_title='Time Bucket',
        yaxis_title='Fraud Rate (%)',
        title_x=0.5,
        width=1200,
        height=600
    )
    fig.show()

#### Monthly Fraud Rate By Bank

In [0]:
# Calculate monthly fraud rate per bank
monthly_fraud_rate = (
    bank_fraud_clean
    .groupby(['month', 'bank'])['is_fraud']
    .mean()
    .reset_index()
)

monthly_fraud_rate['fraud_rate'] = monthly_fraud_rate['is_fraud'] * 100

monthly_fraud_rate['month'] = pd.Categorical(
    monthly_fraud_rate['month'],
    categories=month_order,
    ordered=True
)

monthly_fraud_rate = monthly_fraud_rate.sort_values('month')

# Create pivot table for stacked bars
pivot_df = monthly_fraud_rate.pivot(
    index='month',
    columns='bank',
    values='fraud_rate'
).fillna(0)

import plotly.graph_objects as go

fig = go.Figure()

for bank in pivot_df.columns:
    fig.add_trace(
        go.Bar(
            x=pivot_df.index,
            y=pivot_df[bank],
            name=bank,
            text=[f"{v:.2f}%" if v > 0 else "" for v in pivot_df[bank]],
            textposition='inside'
        )
    )

fig.update_layout(
    barmode='stack',
    title='Monthly Fraud Rate by Bank',
    xaxis_title='Month',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1200,
    height=600,
    legend_title='Bank'
)

fig.show()

#### Monthly Fraud Rte By Bank(Filtered)

#### Monthly Fraud Rate BY Channel

In [0]:
# Calculate monthly fraud rate per channel
monthly_fraud_rate = (
    bank_fraud_clean
    .groupby(['month', 'channel'])['is_fraud']
    .mean()
    .reset_index()
)

monthly_fraud_rate['fraud_rate'] = monthly_fraud_rate['is_fraud'] * 100

month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

monthly_fraud_rate['month'] = pd.Categorical(
    monthly_fraud_rate['month'],
    categories=month_order,
    ordered=True
)

monthly_fraud_rate = monthly_fraud_rate.sort_values('month')

fig = px.line(
    monthly_fraud_rate,
    x='month',
    y='fraud_rate',
    color='channel',
    markers=True,
    labels={'month': 'Month', 'fraud_rate': 'Fraud Rate (%)', 'channel': 'Channel'},
    title='Monthly Fraud Rate Trend by Channel'
)

fig.update_traces(text=[f"{v:.2f}%" for v in monthly_fraud_rate['fraud_rate']], textposition='top center')
fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Fraud Rate (%)',
    title_x=0.5,
    width=1200,
    height=600,
    legend_title='Channel'
)
fig.show()

In [0]:
# Transaction Volume By Channel

channel_counts = bank_fraud_clean["channel"].value_counts().reset_index()
channel_counts.columns = ["channel", "transaction_count"]

fig = px.bar(
    channel_counts,
    x="transaction_count",
    y="channel",
    orientation="h",
    color="channel",
    text="transaction_count",
    labels={"transaction_count": "Number of Transactions", "channel": "Channel"},
    title="Transaction Count by Channel"
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title="Number of Transactions",
    yaxis_title="Channel",
    width=1000,
    height=600,
    showlegend=False
)
fig.show()

In [0]:
# Transaction Volume By Bank

bank_counts = bank_fraud_clean["bank"].value_counts().reset_index()
bank_counts.columns = ["bank", "transaction_count"]

fig = px.bar(
    bank_counts,
    x="transaction_count",
    y="bank",
    orientation="h",
    color="bank",
    text="transaction_count",
    labels={"transaction_count": "Number of Transactions", "bank": "Bank"},
    title="Transaction Count by Bank"
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title="Number of Transactions",
    yaxis_title="Bank",
    width=1000,
    height=600,
    showlegend=False
)
fig.show()

In [0]:
display(
    bank_fraud.groupby(
        ['channel', 'bank', 'merchant_category', 'age_group', 'location']
    )['is_fraud']
    .mean()
    .reset_index()
    .sort_values(by='is_fraud', ascending=False)
)

#### Advnced Analysis

In [0]:
corr_target = (
    bank_fraud_clean.corr(numeric_only=True)['is_fraud']
    .abs()
    .sort_values(ascending=False)
)

import plotly.express as px

fig = px.bar(
    x=corr_target.index[1:15],
    y=corr_target.values[1:15],
    labels={'x': 'Feature', 'y': 'Correlation with Fraud'},
    title='Top Features Associated with Fraud'
)
fig.update_layout(xaxis_title='Feature', yaxis_title='Correlation with Fraud')
fig.show()

In [0]:
#Fraud Funnel Analysis

#Shows where fraud originates.

fraud_funnel = (
    bank_fraud_clean[bank_fraud_clean['is_fraud']==1]
    .groupby('channel')
    .size()
    .reset_index(name='fraud_transactions')
)

fig = px.bar(
    fraud_funnel,
    x='channel',
    y='fraud_transactions',
    text='fraud_transactions',
    labels={'channel': 'Channel', 'fraud_transactions': 'Fraud Transactions'},
    title='Fraud Transactions by Channel'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Channel',
    yaxis_title='Fraud Transactions',
    width=1000,
    height=600,
    showlegend=False
)
fig.show()

In [0]:
#Fraud Rate by Hour and Day

#Extremely powerful feature-engineering visualization.

hour_day = (
    bank_fraud_clean
    .groupby(['hour', 'day_of_week'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

import plotly.express as px

fig = px.density_heatmap(
    hour_day,
    x='day_of_week',
    y='hour',
    z='is_fraud',
    color_continuous_scale='Reds',
    labels={'is_fraud': 'Fraud Rate (%)', 'day_of_week': 'Day of Week', 'hour': 'Hour'},
    title='Fraud Rate by Hour and Day',
    text_auto=True
)

fig.update_traces(hovertemplate='Day: %{x}<br>Hour: %{y}<br>Fraud Rate: %{z:.2f}%')
fig.update_layout(width=1200, height=700)
fig.show()

In [0]:
# Fraud Rate by Age Group and Channel
age_channel = (
    bank_fraud_clean
    .groupby(['age_group', 'channel'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

import plotly.express as px

fig = px.density_heatmap(
    age_channel,
    x='channel',
    y='age_group',
    z='is_fraud',
    color_continuous_scale='Reds',
    labels={'is_fraud': 'Fraud Rate (%)', 'age_group': 'Age Group', 'channel': 'Channel'},
    title='Fraud Rate by Age Group and Channel',
    text_auto=True
)

fig.update_traces(hovertemplate='Age Group: %{y}<br>Channel: %{x}<br>Fraud Rate: %{z:.2f}%')
fig.update_layout(width=1000, height=600)
fig.show()

In [0]:
#Fraud Rate by Location and Channel
location_channel = (
    bank_fraud_clean
    .groupby(['location', 'channel'])['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

import plotly.express as px

fig = px.density_heatmap(
    location_channel,
    x='channel',
    y='location',
    z='is_fraud',
    color_continuous_scale='Reds',
    labels={'is_fraud': 'Fraud Rate (%)', 'location': 'Location', 'channel': 'Channel'},
    title='Fraud Rate by Location and Channel',
    text_auto=True
)

fig.update_traces(hovertemplate='Location: %{y}<br>Channel: %{x}<br>Fraud Rate: %{z:.2f}%')
fig.update_layout(width=1200, height=700)
fig.show()

In [0]:
# KDE Plot for Fraud Separation

# One of the best model-building plots.

import plotly.express as px

fig = px.histogram(
    bank_fraud_clean,
    x='velocity_score',
    color='is_fraud',
    marginal='rug',
    histnorm='density',
    opacity=0.7,
    nbins=50,
    labels={'velocity_score': 'Velocity Score', 'is_fraud': 'Fraud'},
    title='Velocity Score Distribution by Fraud'
)
fig.update_layout(width=1000, height=600)
fig.show()

#Repeat for:

#amount
#merchant_risk_score
#composite_risk
#amount_vs_mean_ratio

In [0]:
import plotly.express as px

fig = px.violin(
    bank_fraud_clean,
    x='is_fraud',
    y='velocity_score',
    box=True,
    points='all',
    color='is_fraud',
    labels={'velocity_score': 'Velocity Score', 'is_fraud': 'Fraud'},
    title='Velocity Score Distribution by Fraud'
)
fig.update_layout(width=1000, height=600)
fig.show()

In [0]:
#SHAP-Style Correlation Ranking

#Before training a model.

corr_target = (
    bank_fraud_clean.corr(numeric_only=True)['is_fraud']
    .abs()
    .sort_values(ascending=False)
)

import plotly.express as px

fig = px.bar(
    x=corr_target.index[1:15],
    y=corr_target.values[1:15],
    labels={'x': 'Feature', 'y': 'Correlation with Fraud'},
    title='Top Features Associated with Fraud'
)
fig.update_layout(xaxis_title='Feature', yaxis_title='Correlation with Fraud')
fig.show()

In [0]:
#32. Customer Risk Segmentation
customer_risk = (
    bank_fraud.groupby('customer_id')['is_fraud']
    .mean()
    .mul(100)
    .reset_index()
)

import plotly.express as px

fig = px.histogram(
    customer_risk,
    x='is_fraud',
    nbins=30,
    labels={'is_fraud': 'Customer Fraud Rate (%)'},
    title='Customer Risk Distribution'
)
fig.update_layout(width=1000, height=600)
fig.show()

In [0]:
bank_fraud_clean

In [0]:
bank_fraud_clean.to_csv('dataset.csv', index=False)


In [0]:
bank_fraud_clean.to_parquet('dataset.parquet', index=False)




# Model Logic

In [0]:
%pip install pycaret

In [0]:
bank_fraud_clean.columns

In [0]:
df_model= bank_fraud_clean.drop(['timestamp','time','online_channel_ratio', 'amount_log', 'DayClassification', 'gold_processed_timestamp','data_quality_score','is_fraud'], axis=1, inplace=True)

In [0]:
fraud_percentage = bank_fraud_clean['is_fraudulent'].mean() * 100
fraud_percentage

## Testing Logistic Regression Model

In [0]:
pip install scikit-learn

In [0]:
# IMPORT SKLEARN LIBRARIES FOR THE Model
from sklearn.model_selection import train_test_split                                     # For splitting data set 80/20
from sklearn.preprocessing import StandardScaler                                         # For scaling, shifts mean to 0 and standard deviation to 1         
from sklearn.linear_model import LogisticRegression                                      # Type of Model, Classification(Yes/N0)
from sklearn.metrics import classification_report, confusion_matrix                      # Model Evalution
from sklearn.pipeline import Pipeline                                                    # For Transformation & Model Training
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder                                          # Encoding in numbers

In [0]:
bank_fraud_clean.columns

In [0]:
banklr_model=bank_fraud_clean.copy()

In [0]:
# Setting The target and Labels
y=banklr_model['is_fraudulent']
X=banklr_model.drop('is_fraudulent', axis=1)

In [0]:
# Assigning Training dataset and testing

X_train,X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, stratify=y)      # test_size is the pliting, splited 80/20

In [0]:
# Numerical Features
numeric = [
    'amount',
    'hour',
    'is_weekend',
    'is_peak_hour',
    'velocity_score',
    'merchant_risk_score',
    'composite_risk',
    'year',
    'day',
    'risk_decile',
    'amount_decile'
]

# Categorical Features
categorical = [
    'channel',
    'merchant_category',
    'bank',
    'location',
    'age_group',
    'day_of_week',
    'month',
    'season',
    'time_bucket',
    'month_category',
   
]

# Target Variable
target = 'is_fraudulent'

In [0]:
# Preprocessing

preprocesser =ColumnTransformer(
    transformers=[
       ('num', StandardScaler(), numeric),
       ('cat', OneHotEncoder(drop='first'), categorical)
    ],
    remainder='drop'
)

In [0]:
# Creating Model Pipeline

pipeline= Pipeline([
    ('prep', preprocesser),
    ('clf',LogisticRegression(class_weight='balanced', max_iter=1000))     # balance the class since 99% of the data is non fraud and the model might detect non fraud
])

In [0]:
# Training the Model

pipeline.fit(X_train, y_train)

In [0]:
# Prediction
pipeline.predict(X_test)
#Or
y_prediction=pipeline.predict(X_test)

In [0]:
# compare y_prediction with y_test
classification_report(y_test,y_prediction)
print(classification_report(y_test,y_prediction))

In [0]:
confusion_matrix(y_test,y_prediction)

## comparing Different Models @ Once Using Pycaret

In [0]:
df_model=bank_fraud_clean.copy()

In [0]:
df_model.columns

In [0]:
labels_model = df_model.drop(columns=["is_fraudulent"])
target_model = df_model["is_fraudulent"]

In [0]:
# Initialize PyCaret with your data
Df_Model = setup(
    data=labels_model,
    target=target_model,
    log_experiment=False,  # disable MLflow to avoid compatibility issues
    session_id=123
)

In [0]:
from pycaret.classification import setup, compare_models

# Set up the experiment
model_setup = setup(
    data=labels_model,
    target=target_model
)
best_model = compare_models()

In [0]:
from pycaret.classification import *
#Create a model
model = create_model('rf')
#Tune the model
tuned_model = tune_model(model)
#Finalize the model
final_model = finalize_model(tuned_model)
#Predict on test set
predictions = predict_model(final_model)
predictions

In [0]:
# Precision-Recall Curve 
plot_model(final_model, plot='pr')


In [0]:

plot_model(final_model, plot='error')


In [0]:
#  what drives fraud predictions
plot_model(final_model, plot='feature')

In [0]:
#  model discrimination ability
plot_model(final_model, plot='auc')

In [0]:
plot_model(final_model, plot='class_report')

In [0]:
plot_model(final_model, plot='confusion_matrix')

In [0]:
evaluate_model(final_model)

In [0]:
pip install mlflow

# Deployment and Monitoring
- Use Final Model to generate predictions on unseen data with the best model

In [0]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import mlflow
import mlflow.sklearn

from pycaret.classification import *

# ==========================================
# 2. CREATE MLFLOW EXPERIMENT
# ==========================================

mlflow.set_experiment(
    "/Users/sshanay92@gmail.com/Bank Fraud Detection"
)
# ==========================================
# 6. SAVE METRICS TABLE
# ==========================================

metrics_df = pull(final_model)

display(metrics_df)

display(predictions.head())

# ==========================================
# 9. EXTRACT METRICS
# ==========================================

accuracy = float(metrics_df['Accuracy'].iloc[0])
auc = float(metrics_df['AUC'].iloc[0])
recall = float(metrics_df['Recall'].iloc[0])
precision = float(metrics_df['Prec.'].iloc[0])
f1 = float(metrics_df['F1'].iloc[0])
kappa = float(metrics_df['Kappa'].iloc[0])
mcc = float(metrics_df['MCC'].iloc[0])

# ==========================================
# 10. START MLFLOW RUN
# ==========================================

with mlflow.start_run(
    run_name="RandomForest_Fraud_Detection"
):

    # Log Metrics
  

    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "auc",
        auc
    )

    mlflow.log_metric(
        "recall",
        recall
    )

    mlflow.log_metric(
        "precision",
        precision
    )

    mlflow.log_metric(
        "f1",
        f1
    )

    mlflow.log_metric(
        "kappa",
        kappa
    )

    mlflow.log_metric(
        "mcc",
        mcc
    )

    # ------------------------------
    # Log Parameters
    # ------------------------------

    mlflow.log_param(
        "model_type",
        "Random Forest"
    )

    mlflow.log_param(
        "target_column",
        "is_fraudulent"
    )

    # ------------------------------
    # Log Model
    # ------------------------------

    mlflow.sklearn.log_model(
        sk_model=final_model,
        name="Random_Forest_Model"
    )

    print("✅ Model Logged Successfully")

    print(
        f"Run ID: {mlflow.active_run().info.run_id}"
    )

# ==========================================
# 11. SAVE MODEL LOCALLY
# ==========================================

save_model(
    final_model,
    "bank_fraud_random_forest"
)

print("✅ Model Saved")

In [0]:
from pycaret.classification import setup

setup(
    data=df_model,
    target='is_fraudulent',  # ← also fixed the column name
    log_experiment=False,  # ← disable MLflow
    session_id=123
)

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# =====================================
# Train Model
# =====================================

model = create_model('rf')
tuned_model = tune_model(model)
results = pull()
final_model = finalize_model(tuned_model)
predictions = predict_model(final_model)

# =====================================
# DEBUG: Check Available Columns
# =====================================

print("Available columns in results:")
print(results.columns.tolist())
print("\nResults DataFrame:")
print(results)


# Extract Metrics (Robust Method)

# Helper function to safely extract metrics
def safe_metric_extract(df, possible_names, default=0.0):
    """Try multiple column name variations"""
    for name in possible_names:
        if name in df.columns:
            return float(df[name].iloc[0])
    print(f"Warning: Could not find any of {possible_names}")
    return default

accuracy = safe_metric_extract(results, ['Accuracy', 'ACC'])
auc = safe_metric_extract(results, ['AUC', 'ROC AUC'])
recall = safe_metric_extract(results, ['Recall', 'Rec'])
precision = safe_metric_extract(results, ['Precision', 'Prec', 'Precision_1'])
f1 = safe_metric_extract(results, ['F1', 'F1-Score'])
kappa = safe_metric_extract(results, ['Kappa'])
mcc = safe_metric_extract(results, ['MCC'])

# =====================================
# MLflow Experiment
# =====================================

mlflow.set_experiment("/Users/sshanay92@gmail.com/Bank Fraud Detection")

# Create sample input for signature
input_example = predictions.drop(
    columns=['prediction_label', 'prediction_score'],
    errors='ignore'
).head(5)

signature = infer_signature(
    input_example,
    final_model.predict(input_example)
)

# =====================================
# Log Run
# =====================================

with mlflow.start_run(run_name="RandomForest_Fraud_Detection"):
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="Random_Forest_Model",
        signature=signature,
        input_example=input_example
    )

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("kappa", kappa)
    mlflow.log_metric("mcc", mcc)

    fraud_percentage = predictions['prediction_label'].mean() * 100
    mlflow.log_metric("fraud_percentage_predicted", fraud_percentage)
    mlflow.log_param("model_type", "Random Forest")

    print("✅ Model successfully logged to MLflow")
    print(f"Run ID: {mlflow.active_run().info.run_id}")